In [1]:
# Statistical Hypothesis Testing
#This notebook tests whether observed business relationships are statistically significant using Welch's t-test, 
# one-way ANOVA, and the chi-square test of independence.

In [4]:
import pandas as pd

from scipy.stats import ttest_ind, f_oneway, chi2_contingency
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

mysql_username = input("MySQL username [root]: ").strip() or "root"
mysql_password = getpass("MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_username,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="cart2insights"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

In [5]:
## Test 1: Delivery Performance vs Review Score

#- H₀: On-time and delayed orders have the same average review score.
#- H₁: Their average review scores are significantly different.
#- Significance level: α = 0.05

In [6]:
delivery_reviews = pd.read_sql(
    """
    WITH review_scores AS (
        SELECT
            order_id,
            AVG(review_score) AS review_score
        FROM reviews
        GROUP BY order_id
    )

    SELECT
        CASE
            WHEN o.order_delivered_customer_date
                 <= o.order_estimated_delivery_date
            THEN 'On Time'
            ELSE 'Delayed'
        END AS delivery_status,
        r.review_score
    FROM orders AS o
    JOIN review_scores AS r
        ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL;
    """,
    engine
)

on_time_scores = delivery_reviews.loc[
    delivery_reviews["delivery_status"] == "On Time",
    "review_score"
].dropna()

delayed_scores = delivery_reviews.loc[
    delivery_reviews["delivery_status"] == "Delayed",
    "review_score"
].dropna()

t_statistic, p_value = ttest_ind(
    on_time_scores,
    delayed_scores,
    equal_var=False
)

print(f"On-time mean: {on_time_scores.mean():.2f}")
print(f"Delayed mean: {delayed_scores.mean():.2f}")
print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.6g}")

if p_value < 0.05:
    print("Decision: Reject H₀")
    print("Conclusion: Review scores differ significantly by delivery status.")
else:
    print("Decision: Fail to reject H₀")
    print("Conclusion: No statistically significant difference was detected.")

On-time mean: 4.29
Delayed mean: 2.57
T-statistic: 89.3844
P-value: 0
Decision: Reject H₀
Conclusion: Review scores differ significantly by delivery status.


In [7]:
# Result
#Welch's t-test found a statistically significant difference between the review scores of on-time and delayed orders (t = 89.38, p < 0.001). On-time orders received an average score of 4.29, compared with 2.57 for delayed orders.
#Therefore, the null hypothesis was rejected. Delivery delays are associated with substantially lower customer satisfaction. The 1.72-point difference is also meaningful from a business perspective.

In [8]:
# Test 2: Product Category vs Spending

#- H₀: Mean spending is equal across the selected product categories.
#- H₁: At least one product category has a different mean spending.
#- Significance level: α = 0.05

In [9]:
category_spending = pd.read_sql(
    """
    SELECT
        o.order_id,
        COALESCE(p.product_category_name, 'unknown') AS product_category,
        SUM(oi.price + oi.freight_value) AS spending
    FROM orders AS o
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    JOIN products AS p
        ON oi.product_id = p.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        o.order_id,
        COALESCE(p.product_category_name, 'unknown');
    """,
    engine
)

top_categories = (
    category_spending["product_category"]
    .value_counts()
    .head(10)
    .index
)

anova_data = category_spending[
    category_spending["product_category"].isin(top_categories)
].copy()

category_groups = [
    group["spending"].dropna().values
    for _, group in anova_data.groupby("product_category")
]

f_statistic, anova_p_value = f_oneway(*category_groups)

category_means = (
    anova_data.groupby("product_category", as_index=False)
    .agg(
        average_spending=("spending", "mean"),
        observations=("spending", "count")
    )
    .sort_values("average_spending", ascending=False)
)

display(category_means)

print(f"F-statistic: {f_statistic:.4f}")
print(f"P-value: {anova_p_value:.6g}")

if anova_p_value < 0.05:
    print("Decision: Reject H₀")
    print("Conclusion: Average spending differs across product categories.")
else:
    print("Decision: Fail to reject H₀")
    print("Conclusion: No significant spending difference was detected.")

,product_category,average_spending,observations
7,relogios_presentes,230.087920,5495
0,automotivo,175.709908,3810
1,beleza_saude,163.303982,8647
5,informatica_acessorios,158.150654,6530
4,esporte_lazer,148.506894,7530
2,brinquedos,143.812056,3804
6,moveis_decoracao,139.579819,6307
3,cama_mesa_banho,132.140774,9272
9,utilidades_domesticas,132.055067,5743
8,telefonia,92.646621,4093


F-statistic: 188.5041
P-value: 0
Decision: Reject H₀
Conclusion: Average spending differs across product categories.


In [10]:
# Result

#The one-way ANOVA found a statistically significant difference in average spending across the ten most frequently purchased product categories (F = 188.50, p < 0.001).

#Therefore, the null hypothesis was rejected. At least one product category has a different average spending level. For example, 
# average spending was highest for `relogios_presentes` at approximately R$230.09 and lowest for `telefonia` at approximately R$92.65.

#ANOVA confirms that a difference exists, but a post-hoc test would be required to identify every specific pair of categories that differs.

In [11]:
# Test 3: Payment Method vs Order Status

#- H₀: Payment method and order status are independent.
#- H₁: Payment method and order status are associated.
#- Significance level: α = 0.05

In [15]:
import numpy as np

payment_status = pd.read_sql(
    """
    SELECT
        p.payment_type,
        o.order_status
    FROM payments AS p
    JOIN orders AS o
        ON p.order_id = o.order_id
    WHERE p.payment_sequential = 1
      AND p.payment_type IS NOT NULL
      AND p.payment_type <> 'not_defined';
    """,
    engine
)

payment_counts = payment_status["payment_type"].value_counts()
status_counts = payment_status["order_status"].value_counts()

valid_payment_types = payment_counts[
    payment_counts >= 500
].index

valid_statuses = status_counts[
    status_counts >= 500
].index

chi_square_data = payment_status[
    payment_status["payment_type"].isin(valid_payment_types)
    & payment_status["order_status"].isin(valid_statuses)
].copy()

contingency_table = pd.crosstab(
    chi_square_data["payment_type"],
    chi_square_data["order_status"]
)

display(contingency_table)

chi2_statistic, chi_p_value, degrees_of_freedom, expected = (
    chi2_contingency(contingency_table)
)

expected_table = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

n = contingency_table.to_numpy().sum()
rows, columns = contingency_table.shape

cramers_v = np.sqrt(
    chi2_statistic
    / (n * min(rows - 1, columns - 1))
)

print(f"Chi-square statistic: {chi2_statistic:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"P-value: {chi_p_value:.6g}")
print(f"Minimum expected count: {expected_table.min().min():.2f}")
print(f"Cramer's V: {cramers_v:.4f}")

if chi_p_value < 0.05:
    print("Decision: Reject H₀")
    print("Conclusion: Payment method and order status are associated.")
else:
    print("Decision: Fail to reject H₀")
    print("Conclusion: No significant association was detected.")

order_status,canceled,delivered,shipped,unavailable
payment_type,,,,
boleto,95,19190,209,150
credit_card,444,74275,848,443
debit_card,7,1436,22,5
voucher,76,1498,28,10


Chi-square statistic: 458.4413
Degrees of freedom: 9
P-value: 4.49393e-93
Minimum expected count: 9.05
Cramer's V: 0.0393
Decision: Reject H₀
Conclusion: Payment method and order status are associated.


In [16]:
### Result

#The chi-square test found a statistically significant association between payment method and order status, χ²(9) = 458.44, p < 0.001.
#Therefore, the null hypothesis was rejected.

#The minimum expected cell count was 9.05, so the chi-square expected-frequency assumption was satisfied. 
#However, Cramer's V was only 0.0393, indicating that the relationship is extremely weak. 
# Payment method alone is therefore unlikely to be an important predictor of order status.